In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt

%matplotlib inline
%matplotlib notebook

## 1.Load file.

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

## 2. ClosedPaved ###

In [4]:
iters = np.shape(P_atm)[0] # total timestep.

In [5]:
CP_measure = 0 # we do not consider 'measure' for the time being.

In [6]:
class ClosedPaved:
    def __init__(self, init_instor, intstorcap_closedpaved = 1.6, stormfrac_closedpaved = 1.0, discfrac_closedpaved = 0.0):
        
        # state
        self.init_instor = init_instor # Give the inital interception storage
        
        # parameters
        self.intstorcap = intstorcap_closedpaved
        self.stormfrac = stormfrac_closedpaved
        self.discfrac = discfrac_closedpaved
        
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are input information.'
        
    def mxd_frac(self):
        return 1 - self.stormfrac
        
    def sol(self, p_atm , e_pot_ow):
        intcp = min(self.intstorcap, max(0, p_atm + self.init_instor))
        e_atm = min(e_pot_ow, intcp)
        intstor = intcp - e_atm
        r_swds = self.stormfrac * (1 - self.discfrac) * max(0, p_atm - e_atm - (intstor - self.init_instor))
        r_mss = self.mxd_frac() * (1 - self.discfrac) * max(0, p_atm - e_atm - (intstor - self.init_instor))
        r_up = self.discfrac * max(0, p_atm - e_atm - (intstor - self.init_instor))
        
        # update state
        self.init_instor = intstor
        
        return intcp, e_atm, intstor, r_swds, r_mss, r_up

In [7]:
t = 1
E_atm = [0]
Intcp = [0]
IntStor = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

# Give initial interception storage.
init_instor_t0 = 0 

# Specify the parameter or use the default setting.
m = ClosedPaved(init_instor_t0, intstorcap_closedpaved = 1.6, stormfrac_closedpaved = 1.0, discfrac_closedpaved = 0.0)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t])
    
    Intcp.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_swds.append(sol[3])
    R_mss.append(sol[4])
    R_up.append(sol[5])
    
    # print('time step', t)
    t += 1
    
filename = 'Results_ClosedPaved.csv'
np.savetxt('sol/' + filename, np.c_[Intcp, E_atm, IntStor, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.


### Conclusion.
The closedpaved structure is the same as the pavedroof structure. 

The parameters of interceptioncap and disconnectedfraction may be different for three paved types in reality and can be specified in python module.

__Note that__ a general stormfrac is taken for all three paved units in excel (same location reference). While in python module, we can specify different stormfrac or use default 1.6 value. Options of general frac or specified frac should be considered in further programming. 

### For data preparation of unpaved unit buildup:
1. C2S1: discfrac = 0.285, stormfrac = 0.75, get R_up:

In [9]:
t = 1
E_atm = [0]
Intcp = [0]
IntStor = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

# Give initial interception storage.
init_instor_t0 = 0 

# Specify the parameter or use the default setting.
m = ClosedPaved(init_instor_t0, intstorcap_closedpaved = 1.6, stormfrac_closedpaved = 0.75, discfrac_closedpaved = 0.285)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t])
    
    Intcp.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_swds.append(sol[3])
    R_mss.append(sol[4])
    R_up.append(sol[5])
    
    # print('time step', t)
    t += 1
    
filename = 'Results_closedpaved_forunpavedbuildup_c2s1_discfrac0285storm075.csv'
np.savetxt('sol/' + filename, np.c_[Intcp, E_atm, IntStor, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.
